In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

In [3]:
three_rt = pd.read_csv(
    "C:\\Users\\jhonm\\Desktop\\Code\\Python\\SFLP\\data\\_inputfiles\\03_RT.csv", sep="\t"
)
three_rt

,CASE,RT,FREQ,FAM,IMA,MEAN
0,almond,650.9947,2,NaN,NaN,NaN
1,ant,589.4347,7,med,hi,415.0
2,apple,523.0493,10,hi,hi,451.0
3,apricot,642.3342,2,lo,lo,NaN
4,asparagus,696.2092,2,med,lo,442.0
...,...,...,...,...,...,...
72,tortoise,733.0323,4,lo,lo,403.0
73,walnut,663.5908,12,med,lo,468.0
74,wasp,725.7056,3,NaN,NaN,NaN
75,whale,609.9745,1,med,hi,474.0


In [4]:
three_rt.describe()

,RT,FREQ,MEAN
count,77.000000,77.000000,48.000000
mean,631.859958,9.259740,435.979167
std,60.163206,16.993344,43.243738
min,523.049300,1.000000,315.000000
25%,589.434700,2.000000,409.750000
50%,617.736700,4.000000,437.500000
75%,663.590800,10.000000,466.250000
max,794.452200,118.000000,553.000000


In [5]:
three_rt["FAM"] = pd.Categorical(three_rt["FAM"], categories=["lo", "med", "hi"], ordered=True)
three_rt["IMA"] = pd.Categorical(three_rt["IMA"], categories=["lo", "hi"], ordered=True)
three_rt

,CASE,RT,FREQ,FAM,IMA,MEAN
0,almond,650.9947,2,NaN,NaN,NaN
1,ant,589.4347,7,med,hi,415.0
2,apple,523.0493,10,hi,hi,451.0
3,apricot,642.3342,2,lo,lo,NaN
4,asparagus,696.2092,2,med,lo,442.0
...,...,...,...,...,...,...
72,tortoise,733.0323,4,lo,lo,403.0
73,walnut,663.5908,12,med,lo,468.0
74,wasp,725.7056,3,NaN,NaN,NaN
75,whale,609.9745,1,med,hi,474.0


## Categorical Values

In [6]:
three_rt["IMA"].value_counts(dropna=False)

IMA
hi     30
NaN    24
lo     23
Name: count, dtype: int64

### Observed Percentages

In [7]:
three_rt["IMA"].value_counts(dropna=False, normalize=True)


IMA
hi     0.389610
NaN    0.311688
lo     0.298701
Name: proportion, dtype: float64

### Mode

In [8]:
three_rt["IMA"].value_counts().sort_values(ascending=False).reset_index(drop=True)[0]

np.int64(30)

In [9]:
letters = ["a", "b", "c", "d", "e"]
repeats = [3, 5, 7, 7, 5]
some_factor = np.repeat(letters, repeats)
some_factor = pd.Series(some_factor, dtype="category")
freq_table = some_factor.value_counts().sort_index()
freq_table

a    3
b    5
c    7
d    7
e    5
Name: count, dtype: int64

If we're supposed to tell someone as much as possible about IMA but are allowed to state only one number, then our smartest choice is to tell them the frequency of the most frequent level, and that's what's called the **mode**: 

In [10]:
max(freq_table)

7

Again, though: be ready for 'the unexpected': What does our approach do if two or more levels are equally maximally frequent? 

In [11]:
freq_table.nlargest(1, keep="all")

c    7
d    7
Name: count, dtype: int64

### Dispersion: normalized entropy

As mentioned above, any measure of central tendency should always – as in always! – be accompanied by a measure of dispersion because otherwise we're not telling our readers how much information we have provided them with, or how good the information is we provided when we gave them our measure of central tendency. Thus, always – in case I didn't mention that before – provide a measure of dispersion. One measure we can use for categorical variables is a normalized entropy, written as $H_{norm}$. For a categorical variable with $k$ different levels with observed percentages $p_1$ to $p_k$, $H_{norm}$ is computed like this:

$$
H_{norm} = -\frac{\sum_{i=1}^{k} p_i \times \log_2 p_i}{\log_2 k}
$$

*(where $k$ is the number of categories and $p_i$ is the proportion of data in the $i$-th category).*

Normalized entropy ($H_n$) is a statistical metric scaled between $0$ and $1$ that quantifies the dispersion, diversity, or uncertainty within a categorical variable. It is calculated by taking the standard Shannon entropy and dividing it by the maximum possible entropy for the given number of categories:

**0.1 to 0.2: Extreme Dominance (Near-Zero Dispersion)**
* **Characteristics:** This range indicates a near-monopoly. A single category contains the vast majority of the observations (typically upwards of 95% at 0.1, and 85–90% at 0.2).
* **Interpretation:** Predictability is nearly absolute; random selection of a data point will almost certainly yield the dominant category. In data science, features in this range are frequently classified as having "near-zero variance." They are often excluded from machine learning models because they lack the necessary diversity to differentiate between outcomes.

**0.3 to 0.4: Emerging Minorities (Low Dispersion)**
* **Characteristics:** The dominant category still dictates the variable's overall behavior, encompassing roughly 75% to 80% of the distribution. However, minority categories have grown large enough to be statistically observable rather than functioning merely as noise or outliers.
* **Interpretation:** Confidence in predicting the majority class remains high, but dispersion is gradually increasing. In a linguistic corpus, for instance, this might represent a grammatical structure that is heavily standardized but permits occasional, deliberate deviations.

**0.5 to 0.6: The Skewed Middle (Moderate Dispersion)**
* **Characteristics:** Because entropy is calculated logarithmically, a score of 0.5 does not represent a balanced distribution; it still reflects a distinct skew. In a three-category system, an entropy of 0.5 roughly equates to an 80/10/10 split. By 0.6, the dominant category may account for 65% to 70% of the data.
* **Interpretation:** This range marks a transitional turning point. Minority categories now hold enough weight to introduce genuine uncertainty. The variable shifts from being highly skewed to moderately diverse. Blindly predicting the majority class will begin to yield a significant error rate.

**0.7 to 0.8: Strong Competition (High Dispersion)**
* **Characteristics:** No single category maintains a decisive majority. In a binary system, this corresponds to roughly a 75/25 to 65/35 split. In systems with multiple categories, the largest group might hold only 30% to 40% of the data.
* **Interpretation:** The variable is highly dispersed and information-rich, meaning predictability based on a single dominant class fails. Variables within this range are highly valuable for analytical models as they represent active, healthy variance within a dataset.

**0.9: The Edge of Chaos (Extreme Dispersion)**
* **Characteristics:** The categories are distributed relatively evenly, competing for equal footing. The proportions across categories are highly similar.
* **Interpretation:** Uncertainty is the defining characteristic of the variable. For example, in a three-category system, proportions such as 39%, 31%, and 30% are almost completely flat, resulting in a score near 0.9.

**1.0: Perfect Balance (Maximum Dispersion)**
* **Characteristics:** This indicates perfect uniformity. Every category occurs with the exact same frequency (e.g., in a three-category system, an exact 33.3% / 33.3% / 33.3% split).
* **Interpretation:** This represents maximum dispersion, maximum uncertainty, and zero predictability based on class imbalances. The variable provides no inherent baseline bias.



In [12]:
percentages = three_rt["IMA"].value_counts(normalize=True)
shannon_entropy = stats.entropy(percentages, base=2)
k = len(percentages)
max_entropy = np.log2(k)
normalised_entropy = shannon_entropy / max_entropy

print(f"Categories (k): {k}")
print(f"Shannon Entropy: {shannon_entropy:.4f} bits")
print(f"Max Entropy:     {max_entropy:.4f} bits")
print(f"Normalised Entropy: {normalised_entropy:.4f}")

Categories (k): 2
Shannon Entropy: 0.9874 bits
Max Entropy:     1.0000 bits
Normalised Entropy: 0.9874
